# 7. L2any L2both

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.preprocessing import normalize

mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")
from ALLCools.mcds import RegionDS


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
# adata = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad')
meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', header=0, index_col=0, sep='\t')
meta

In [ ]:
# tmp = meta[['L2_any', 'L2_both']].drop_duplicates()
# tmp['L2_both'] = tmp['L2_both'].str.replace('-c','-b')


In [ ]:
dmr_group = {}
for l1,l1_df in meta.groupby('L1_annot'):
    count = len(l1_df['L2_both'].unique())
    if count>1:
        dmr_group[l1] = {}
        for l2b,l2b_df in l1_df.groupby('L2_both'):
            tmp = list(l2b_df['L2_any'].unique())
            if len(tmp)>1:
                dmr_group[l1][l2b.replace('-c','-b')] = tmp


In [ ]:
ngroup = np.sum([len(dmr_group[xx]) for xx in dmr_group])
print(ngroup)

In [ ]:
import cooler
chrom_size_path = '/large_experiments/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
ncol = 5
nrow = (ngroup - 1) // ncol + 1

k = 0
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow), constrained_layout=True)
for l1 in dmr_group:
    for l2b in dmr_group[l1]:
        ax = axes.flatten()[k]
        k += 1
        if not os.path.exists(f'{indir}DMR/L2both-L2any/{l2b}_dmr'):
            continue
        dmr_ds = RegionDS.open(f'{indir}DMR/L2both-L2any/{l2b}_dmr', region_dim='dmr')
        dmr = dmr_ds[['dmr_chrom', 'dmr_start', 'dmr_end', 'dmr_ndms']].to_pandas()
        dmr[['dmr_start', 'dmr_end', 'dmr_ndms']] = dmr[['dmr_start', 'dmr_end', 'dmr_ndms']].astype(int)
        seldmr = (dmr['dmr_ndms']>1) & (dmr['dmr_chrom'].isin(chrom_sizes.index))
        dmr_ds = dmr_ds.sel({'dmr':dmr.index[seldmr]})
        data = dmr_ds['dmr_da_frac'].to_pandas().fillna(1).T

        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xlim([-0.5, data.shape[1]-0.5])
        ax.set_xticklabels(data.columns, rotation=90)
        ax.set_yticks([])
        ax.set_ylabel(f'{data.shape[0]} DMRs')
        ax.set_title(f'{l1} {l2b}')

        np.random.seed(0)
        if data.shape[0]>5000:
            sel = np.random.choice(np.arange(data.shape[0]), 5000, False)
        elif data.shape[0]>500:
            sel = np.arange(data.shape[0])
        else:
            continue
        cg = sns.clustermap(data.iloc[sel], cmap='Greens_r', metric='cosine', figsize=(0.1,0.1))
        tmp = data.iloc[sel].iloc[cg.dendrogram_row.reordered_ind, cg.dendrogram_col.reordered_ind]
        ax.imshow(tmp, aspect='auto', cmap='Greens_r', vmin=0, vmax=1, interpolation='none', rasterized=True)
        ax.set_xticklabels(tmp.columns, rotation=90)

for ax in axes.flatten()[k:]:
    ax.axis('off')

fig.savefig('L2any_L2both/L2any_L2both_DMR.pdf', transparent=True)


In [ ]:
from scipy.stats import norm, zscore

thres1 = norm.isf(0.025)
thres2 = norm.isf(0.15)
print(thres1, thres2)


In [ ]:
ncol = 5
nrow = (ngroup - 1) // ncol + 1
fdrthres = 1e-3
k = 0
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow), constrained_layout=True)
for l1 in dmr_group:
    for l2b in dmr_group[l1]:
        ax = axes.flatten()[k]
        k += 1
        tmpdir = f'{indir}analysis/L2any_L2both_diffloop/{l2b}/'
        loopq = pd.read_hdf(f'{tmpdir}loop_Q.hdf', key='data')
        loopall = pd.read_hdf(f'{tmpdir}merged_loop.hdf', key='data')
        
        statfilter = (zscore(loopall['Qanova'])>thres2) & (zscore(loopall['Tanova'])>thres2)
        fdrfilter = (loopall['Qfdr']<fdrthres) & (loopall['Efdr']<fdrthres) & (loopall['Tfdr']<fdrthres)
        selloop = statfilter & fdrfilter
        print(l2b, statfilter.sum(), fdrfilter.sum(), selloop.sum())
        data = loopq.loc[selloop]
        
        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xlim([-0.5, data.shape[1]-0.5])
        ax.set_xticklabels(data.columns, rotation=90)
        ax.set_yticks([])
        ax.set_ylabel(f'{data.shape[0]} Diff Loops')
        ax.set_title(f'{l1} {l2b}')
        
        np.random.seed(0)
        if data.shape[0]>5000:
            sel = np.random.choice(np.arange(data.shape[0]), 5000, False)
        elif data.shape[0]>500:
            sel = np.arange(data.shape[0])
        else:
            continue
        cg = sns.clustermap(zscore(data.iloc[sel], axis=1), cmap='Oranges', metric='cosine', figsize=(0.1,0.1))
        tmp = data.iloc[sel].iloc[cg.dendrogram_row.reordered_ind, cg.dendrogram_col.reordered_ind]
        ax.imshow(zscore(tmp, axis=1), aspect='auto', cmap='Oranges', vmin=0, vmax=2, interpolation='none', rasterized=True)
        ax.set_xticklabels(tmp.columns, rotation=90)
    
for ax in axes.flatten()[k:]:
    ax.axis('off')


In [ ]:
ncol = 5
nrow = (ngroup - 1) // ncol + 1
fdrthres = 1e-1
k = 0
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 3*nrow), constrained_layout=True)
for l1 in dmr_group:
    for l2b in dmr_group[l1]:
        ax = axes.flatten()[k]
        k += 1
        tmpdir = f'{indir}analysis/L2any_L2both_diffloop/{l2b}/'
        loopq = pd.read_hdf(f'{tmpdir}loop_Q.hdf', key='data')
        loopall = pd.read_hdf(f'{tmpdir}merged_loop.hdf', key='data')
        
        statfilter = (zscore(loopall['Qanova'])>thres2) & (zscore(loopall['Tanova'])>thres2)
        fdrfilter = (loopall['Qfdr']<fdrthres) & (loopall['Efdr']<fdrthres) & (loopall['Tfdr']<fdrthres)
        selloop = statfilter & fdrfilter
        print(l2b, statfilter.sum(), fdrfilter.sum(), selloop.sum())
        data = loopq.loc[selloop]
        
        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xlim([-0.5, data.shape[1]-0.5])
        ax.set_xticklabels(data.columns, rotation=90)
        ax.set_yticks([])
        ax.set_ylabel(f'{data.shape[0]} Diff Loops')
        ax.set_title(f'{l1} {l2b}')

        np.random.seed(0)
        if data.shape[0]>5000:
            sel = np.random.choice(np.arange(data.shape[0]), 5000, False)
        elif data.shape[0]>500:
            sel = np.arange(data.shape[0])
        else:
            continue
        cg = sns.clustermap(zscore(data.iloc[sel], axis=1), cmap='Oranges', metric='cosine', figsize=(0.1,0.1))
        tmp = data.iloc[sel].iloc[cg.dendrogram_row.reordered_ind, cg.dendrogram_col.reordered_ind]
        ax.imshow(zscore(tmp, axis=1), aspect='auto', cmap='Oranges', vmin=0, vmax=2, interpolation='none', rasterized=True)
        ax.set_xticklabels(tmp.columns, rotation=90)
    
for ax in axes.flatten()[k:]:
    ax.axis('off')

fig.savefig('L2any_L2both/L2any_L2both_diffloop.pdf', transparent=True)


In [ ]:
L1annot = meta[['L1','L1_annot']].drop_duplicates().set_index('L1')['L1_annot']


In [ ]:
cat <(for i in `seq 0 34`; do echo c${i} $(bedtools intersect -wa -a L1-L2both/c${i}_dmr/dmr.bed -b L1-L2any/c${i}_dmr/dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(cat L1-L2both/c${i}_dmr/dmr.bed | wc -l); done) > L2both_L2anyratio.txt
cat <(for i in `seq 0 34`; do echo c${i} $(bedtools intersect -wa -b L1-L2both/c${i}_dmr/dmr.bed -a L1-L2any/c${i}_dmr/dmr.bed | sort -k1,1 -k2,2n -u | wc -l) $(cat L1-L2any/c${i}_dmr/dmr.bed | wc -l); done) > L2any_L2bothratio.txt



In [ ]:
tmp = pd.read_csv(f'{indir}DMR/L2any_L2bothratio.txt', index_col=0, header=None, sep=' ')
tmp.index = tmp.index.map(L1annot)
tmp['ratio'] = tmp[1]/tmp[2]
tmp

In [ ]:
tmp = pd.read_csv(f'{indir}DMR/L2both_L2anyratio.txt', index_col=0, header=None, sep=' ')
tmp.index = tmp.index.map(L1annot)
tmp['ratio'] = tmp[1]/tmp[2]
tmp

In [ ]:
dmr_ds = RegionDS.open(f'{indir}DMR/L1-L2both/c2_dmr', region_dim='dmr')
dmr_ds

In [ ]:
result = []
for l1,l1name in meta[['L1','L1_annot']].drop_duplicates().values:
    l1annot = l1name.replace('/','_').replace(' ','-')
    corr_path_l2both = f'{indir}analysis/L2any_L2both/dmr_loop_corr/L2both/{l1}_{l1annot}_corr.npz'
    corr_path_l2any = f'{indir}analysis/L2any_L2both/dmr_loop_corr/L2any/{l1}_{l1annot}_corr.npz'
    if not os.path.exists(corr_path_l2both):
        continue
    corr_l2both = np.load(corr_path_l2both)
    corr_l2any = np.load(corr_path_l2any)
    result.append([l1, l1annot, 
                   np.mean(corr_l2both['corr']), np.mean(corr_l2any['corr']), 
                   np.mean(corr_l2both['corr_colnorm']), np.mean(corr_l2any['corr_colnorm'])])

result = pd.DataFrame(result, columns=['L1', 'L1annot', 
                                       'corr_L2both', 'corr_L2any', 
                                       'corr_colnorm_L2both', 'corr_colnorm_L2any']).set_index('L1')


In [ ]:
corr_l2any_dmr = []
corr_l2any_bin = []

for l1,l1name in meta[['L1','L1_annot']].drop_duplicates().values:
    l1annot = l1name.replace('/','_').replace(' ','-')
    corr_path_l2any = f'{indir}analysis/L2any_L2both/dmr_loop_corr/L2any/{l1}_{l1annot}_corr.npz'    
    corr_l2any_tmp = np.load(corr_path_l2any)
    corr_l2any_tmp = pd.DataFrame(corr_l2any_tmp['corr'][:, None], columns=['corr'])
    corr_l2any_tmp['L1'] = l1
    corr_l2any_tmp['L1annot'] = l1name
    corr_l2any_dmr.append(corr_l2any_tmp)
    corr_path_l2any = f'{indir}analysis/L2any_L2both/dmr_loop_corr/L2any/{l1}_{l1annot}_10kmcg_corr.npz'    
    corr_l2any_tmp = np.load(corr_path_l2any)
    corr_l2any_tmp = pd.DataFrame(corr_l2any_tmp['corr'][:, None], columns=['corr'])
    corr_l2any_tmp['L1'] = l1
    corr_l2any_tmp['L1annot'] = l1name
    corr_l2any_bin.append(corr_l2any_tmp)

corr_l2any_dmr = pd.concat(corr_l2any_dmr, axis=0)
corr_l2any_bin = pd.concat(corr_l2any_bin, axis=0)



In [ ]:
from sklearn.metrics import adjusted_rand_score as ARI

labelmc, label3c, score = [], [], {}
for l1,l1name in meta[['L1','L1_annot']].drop_duplicates().values:
    l1name = l1name.replace(' ','-').replace('/','_')
    adata = anndata.read_h5ad(f'{indir}clustering/merged/L2/{l1name}/5kCG100k3C_embed.h5ad')
    tmp1 = np.load(f'{indir}clustering/merged/L2_hiconly/{l1name}/mergemcg_rocpr.npy', allow_pickle=True)
    labelmc.append(pd.Series(tmp1, index=adata.obs.index))
    tmp2 = np.load(f'{indir}clustering/merged/L2_hiconly/{l1name}/mergehic_rocpr.npy', allow_pickle=True)
    label3c.append(pd.Series(tmp2, index=adata.obs.index))
    score[l1] = ARI(tmp1, tmp2)
    
labelmc = pd.concat(labelmc)
label3c = pd.concat(label3c)
score = pd.Series(score)


In [ ]:
result['ARI'] = score.copy()


In [ ]:
corr_l2any.groupby('L1')['corr'].median().sort_values().index

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10,5), dpi=300)

ax = axes[0]
legorder = corr_l2any_dmr.groupby('L1')['corr'].median().sort_values().index
sns.violinplot(corr_l2any_dmr, x='L1', y='corr', palette=colors.to_dict(), ax=ax, order=legorder)
ax.set_xticklabels(L1annot.loc[legorder], rotation=90)
ax.plot([0,len(legorder)-1], [0, 0], 'r')
ax.set_xlim([-0.5, len(legorder)-0.5])

ax = axes[1]
legorder = corr_l2any_bin.groupby('L1')['corr'].median().sort_values().index
sns.violinplot(corr_l2any_bin, x='L1', y='corr', palette=colors.to_dict(), ax=ax, order=legorder)
ax.set_xticklabels(L1annot.loc[legorder], rotation=90)
ax.plot([0,len(legorder)-1], [0, 0], 'r')
ax.set_xlim([-0.5, len(legorder)-0.5])

plt.tight_layout()


In [ ]:
colors = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)['color']
colors

In [ ]:
real_d = {}
for f in glob('L2any_L2both/dmr_loop_enrich/L2any/c*_score_bylevel.npz'):
    cluster = f.split('/')[-1].replace('_score_bylevel.npz','')
    # name = L1annot[cluster]
    overlap = np.load(f)
    real_d[cluster] =list(overlap['score']) 


In [ ]:
sns.clustermap(score, cmap='vlag', metric='cosine', col_cluster=False, figsize=(4,7), yticklabels=1)


In [ ]:
from scipy.stats import pearsonr

score = pd.DataFrame.from_dict(real_d, orient='index').fillna(0)
# corr = np.array([pearsonr(np.arange(10), xx)[0] for xx in score.values])
corr = score.values[:,0]
# sel = score.mean(axis=1).sort_values().index[::-1]
score = score.iloc[np.argsort(corr)]

fig, ax = plt.subplots(figsize=(3,5), dpi=300)
ax.imshow(score, cmap='vlag',  aspect='auto',vmin=0.65,vmax=1.35, interpolation='none',rasterized=True)
ax.set_title('DMR enrichment at differential loops', fontsize=10)
ax.set_xticks(np.arange(score.shape[1]))
ax.set_xticklabels([(i+1)/10 for i in range(10)],fontsize=8)
ax.set_yticks(np.arange(score.shape[0]))
ax.set_yticklabels(score.index.map(L1annot),fontsize=8)


In [ ]:
enrich = pd.Series({xx: pearsonr(np.arange(10), yy)[0] for xx,yy in zip(score.index,score.values)})
enrich.sort_values()

In [ ]:
corr = corr_l2any_dmr.groupby('L1')['corr'].mean()
corr.sort_values()

In [ ]:
result = pd.concat([enrich, corr], axis=1)
result.columns = ['DMR diffloop enrich', 'DMR diffloop corr']
result['L1annot'] = result.index.map(L1annot)

In [ ]:
xl, yl = 'DMR diffloop enrich', 'DMR diffloop corr'
fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    

In [ ]:
xl, yl = 'corr_L2both', 'corr_L2any'
fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    

In [ ]:
xl, yl = 'corr_colnorm_L2both', 'corr_colnorm_L2any'
fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    

In [ ]:
xl, yl = 'corr_L2both', 'corr_colnorm_L2both'

fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    

In [ ]:
xl, yl = 'corr_L2any', 'corr_colnorm_L2any', 

fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    

In [ ]:
xl, yl = 'corr_L2any', 'ARI', 

fig, ax = plt.subplots()
ax.scatter(result[xl], result[yl], c=result.index.map(colors))
for xx,yy,l1 in result[[xl, yl, 'L1annot']].values:
    ax.text(xx, yy, l1)
    